In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
# Step 2: Load the California housing dataset
df = pd.read_excel("housing.csv.xlsx")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


In [3]:
# Step 3: Create income categories for stratified sampling based on median_income
# Ensures all income brackets are proportionally represented in train and test sets
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [4]:
# Step 4: Perform Stratified Shuffle Split (80% train, 20% test)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_index].copy()
    strat_test_set = df.loc[test_index].copy()

# Drop temporary 'income_cat' column so the dataset reverts to its original features
for sett in (strat_train_set, strat_test_set):
    sett.drop("income_cat", axis=1, inplace=True)

In [5]:
# Step 5: Separate predictors (housing) and target labels (housing_labels)
# CRITICAL RULE 1: We NEVER scale the target variable ('median_house_value') together with predictors!
# Predictors must predict real house values in dollars ($), not an arbitrary scaled float.
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
12655,-121.46,38.52,29,3873,797.0,2237,706,2.1736,INLAND
15502,-117.23,33.09,7,5320,855.0,2015,768,6.3373,NEAR OCEAN
2908,-119.04,35.37,44,1618,310.0,667,300,2.8750,INLAND
14053,-117.13,32.75,24,1877,519.0,898,483,2.2264,NEAR OCEAN
20496,-118.70,34.28,27,3536,646.0,1837,580,4.4964,<1H OCEAN


In [6]:
# Step 6: Isolate ONLY numerical features
# CRITICAL RULE 2: Feature scaling is strictly for continuous NUMERICAL features!
# We exclude 'ocean_proximity' (categorical text) because dummy variables (0 and 1) should NEVER be scaled.
# Scaling dummy variables destroys their binary meaning and turns them into arbitrary floats.
housing_num = housing.select_dtypes(include=[np.number])
print("Numerical features count:", housing_num.shape[1])
print("Numerical column names:", list(housing_num.columns))
housing_num.describe()

Numerical features count: 8
Numerical column names: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
count,16512.000000,16512.000000,16512.000000,16512.000000,16354.000000,16512.000000,16512.000000,16512.000000
mean,-119.575635,35.639314,28.653404,2622.539789,534.914639,1419.687379,497.011810,3.875884
std,2.001828,2.137963,12.574819,2138.417080,412.665649,1115.663036,375.696156,1.904931
min,-124.350000,32.540000,1.000000,6.000000,2.000000,3.000000,2.000000,0.499900
25%,-121.800000,33.940000,18.000000,1443.000000,295.000000,784.000000,279.000000,2.566950
50%,-118.510000,34.260000,29.000000,2119.000000,433.000000,1164.000000,408.000000,3.541550
75%,-118.010000,37.720000,37.000000,3141.000000,644.000000,1719.000000,602.000000,4.745325
max,-114.310000,41.950000,52.000000,39320.000000,6210.000000,35682.000000,5358.000000,15.000100


In [7]:
# Step 7: Why Feature Scaling is necessary
# Look at the summary statistics above:
# 'total_rooms' ranges up to 39,320!
# 'median_income' ranges from 0.5 to 15.0!
# Machine Learning algorithms (Linear Regression, SVMs, KNN, Neural Networks, Gradient Descent)
# will assume 'total_rooms' is far more important simply because the numbers are larger.
# Feature scaling ensures all features contribute proportionally to distance and weight calculations.
print(f"total_rooms min: {housing_num['total_rooms'].min()}, max: {housing_num['total_rooms'].max()}")
print(f"median_income min: {housing_num['median_income'].min()}, max: {housing_num['median_income'].max()}")

total_rooms min: 6, max: 39320
median_income min: 0.4999, max: 15.0001


In [8]:
# Step 8: Handle missing values first (SimpleImputer)
# Scalers cannot process missing (NaN) values (e.g. in 'total_bedrooms')
# We fit the imputer on housing_num using the median strategy
imputer = SimpleImputer(strategy="median")
housing_num_imputed = imputer.fit_transform(housing_num)
print("Missing values remaining after imputation:", np.isnan(housing_num_imputed).sum())

Missing values remaining after imputation: 0


In [9]:
# Step 9: Technique 1 - Min-Max Scaling (Normalization)
# Formula: x_scaled = (x - x_min) / (x_max - x_min)
# Shifts and rescales all values into a fixed range [0, 1]
# LIMITATION: Highly sensitive to extreme outliers!
# (If one house has 30,000 rooms, all typical houses with 1,000-2,000 rooms get crushed into 0.01 - 0.05)
min_max_scaler = MinMaxScaler(feature_range=(0, 1))
housing_minmax = min_max_scaler.fit_transform(housing_num_imputed)

housing_minmax_df = pd.DataFrame(housing_minmax, columns=housing_num.columns, index=housing_num.index)
print("Min-Max scaled features preview (all values bound between 0 and 1):")
housing_minmax_df.head()

Min-Max scaled features preview (all values bound between 0 and 1):


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
12655,0.287849,0.635494,0.549020,0.098362,0.128061,0.062614,0.131441,0.115426
15502,0.709163,0.058448,0.117647,0.135168,0.137403,0.056392,0.143017,0.402574
2908,0.528884,0.300744,0.843137,0.041003,0.049613,0.018610,0.055639,0.163798
14053,0.719124,0.022317,0.450980,0.047591,0.083280,0.025085,0.089806,0.119067
20496,0.562749,0.184910,0.509804,0.089790,0.103737,0.051403,0.107916,0.275617


In [10]:
# Step 10: Technique 2 - Standardization (StandardScaler)
# Formula: x_scaled = (x - mean) / std_deviation
# Centers values around zero (mean = 0) with unit variance (standard deviation = 1)
# ADVANTAGE: Much less affected by outliers than Min-Max Scaling
# It does not restrict values to a fixed range, making it the default choice for most ML models!
std_scaler = StandardScaler()
housing_std = std_scaler.fit_transform(housing_num_imputed)

housing_std_df = pd.DataFrame(housing_std, columns=housing_num.columns, index=housing_num.index)
print("Standardized features preview:")
housing_std_df.head()

Standardized features preview:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
12655,-0.941350,1.347438,0.027564,0.584777,0.640371,0.732602,0.556286,-0.893647
15502,1.171782,-1.192440,-1.722018,1.261467,0.781561,0.533612,0.721318,1.292168
2908,0.267581,-0.125972,1.220460,-0.469773,-0.545138,-0.674675,-0.524407,-0.525434
14053,1.221738,-1.351474,-0.370069,-0.348652,-0.036367,-0.467617,-0.037297,-0.865929
20496,0.437431,-0.635818,-0.131489,0.427179,0.272790,0.374060,0.220898,0.325752


In [11]:
# Step 11: Verify Standardization properties
# Every standardized feature should have mean ≈ 0 and standard deviation ≈ 1
print("Feature Means (should be ~0.0):")
print(housing_std_df.mean().round(4))
print("\nFeature Standard Deviations (should be ~1.0):")
print(housing_std_df.std().round(4))

Feature Means (should be ~0.0):
longitude             0.0
latitude              0.0
housing_median_age    0.0
total_rooms          -0.0
total_bedrooms       -0.0
population           -0.0
households           -0.0
median_income        -0.0
dtype: float64

Feature Standard Deviations (should be ~1.0):
longitude             1.0
latitude              1.0
housing_median_age    1.0
total_rooms           1.0
total_bedrooms        1.0
population            1.0
households            1.0
median_income         1.0
dtype: float64


In [12]:
# Step 12: CRITICAL RULE 3 - Prevent Data Leakage!
# Always fit scalers ONLY on the training data:
# 1. scaler.fit_transform(X_train)  <-- learns mean and std on train set ONLY
# 2. scaler.transform(X_test)        <-- NEVER call fit() on test data! Only transform()!
#
# Fitting on test data lets the model cheat by peeking at the test set's distribution.